<b><h1 style ="text-align:center; color:gold">Book Recommendation System</h2><b>

### Importing all Necessary Libraries

In [68]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors
from sklearn.feature_extraction.text import TfidfVectorizer

In [69]:
df = pd.read_csv('books.csv')
df.head()

,Rank,Title,Author,Category,Sub-Genre,Format,Price (USD),Rating,Reviews,Weeks on List,Publisher,Year Published,ISBN,Amazon BSR,Amazon URL
0,1,The Let Them Theory,Mel Robbins,Non-Fiction,Self-Help,Paperback,14.99,4.7,98000.0,52.0,Hay House,2024.0,978-1-13278-859-4,112.0,View
1,2,The Dinner Party,Freida McFadden,Fiction,Psychological Thriller,Paperback,11.69,4.3,1800.0,12.0,Sourcebooks Landmark,2025.0,978-3-76237-716-0,175.0,View
2,3,Project Hail Mary,Andy Weir,Fiction,Science Fiction,Paperback,12.99,4.8,130000.0,108.0,Ballantine Books,2021.0,978-4-30379-320-5,64.0,View
3,4,The Correspondent: A Novel,Virginia Evans,Fiction,Literary Fiction,Hardcover,19.58,4.6,87363.0,28.0,Random House,2025.0,978-6-20328-665-4,143.0,View
4,5,Atomic Habits,James Clear,Non-Fiction,Self-Help / Productivity,Paperback,16.99,4.8,120000.0,220.0,Avery,2018.0,978-7-93320-954-5,237.0,View


### Checking the Dataset Shape, Duplicates and NaN Values

In [70]:
df.shape

(500, 15)

In [71]:
df.isna().sum()

Rank              0
Title             0
Author            0
Category          0
Sub-Genre         0
Format            0
Price (USD)       0
Rating            0
Reviews           0
Weeks on List     0
Publisher         0
Year Published    0
ISBN              0
Amazon BSR        0
Amazon URL        0
dtype: int64

In [72]:
df.duplicated().sum()

np.int64(0)

### EDA

In [73]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Rank            500 non-null    int64  
 1   Title           500 non-null    object 
 2   Author          500 non-null    object 
 3   Category        500 non-null    object 
 4   Sub-Genre       500 non-null    object 
 5   Format          500 non-null    object 
 6   Price (USD)     500 non-null    float64
 7   Rating          500 non-null    float64
 8   Reviews         500 non-null    float64
 9   Weeks on List   500 non-null    float64
 10  Publisher       500 non-null    object 
 11  Year Published  500 non-null    float64
 12  ISBN            500 non-null    object 
 13  Amazon BSR      500 non-null    float64
 14  Amazon URL      500 non-null    object 
dtypes: float64(6), int64(1), object(8)
memory usage: 58.7+ KB


In [74]:
df.describe(include='object')

,Title,Author,Category,Sub-Genre,Format,Publisher,ISBN,Amazon URL
count,500,500,500,500,500,500,500,500
unique,500,168,2,48,5,56,500,1
top,The Ice of Stars,Taylor Jenkins Reid,Fiction,Science Fiction,Paperback,Scholastic Press,978-5-65584-318-9,View
freq,1,8,254,21,199,17,1,500


In [75]:
df.describe(include='number')

,Rank,Price (USD),Rating,Reviews,Weeks on List,Year Published,Amazon BSR
count,500.000000,500.000000,500.000000,500.000000,500.000000,500.000000,500.000000
mean,250.500000,16.828040,4.412200,12605.154000,25.106000,2021.032000,3862.358000
std,144.481833,6.801869,0.255032,24759.378719,68.100294,5.116183,2368.629814
min,1.000000,4.230000,3.600000,85.000000,1.000000,1965.000000,30.000000
25%,125.750000,11.207500,4.200000,1952.750000,6.000000,2019.000000,1789.750000
50%,250.500000,15.945000,4.400000,3893.000000,12.000000,2022.000000,3735.000000
75%,375.250000,21.930000,4.600000,10986.500000,22.000000,2024.000000,5555.250000
max,500.000000,34.090000,5.000000,220000.000000,1200.000000,2026.000000,9600.000000


### Quick Observation

The higest cost of a book is $34.09, the average price of book is $15.94 and the lowest is $4.23.
The top author is Taylor Jenkins Reid	and top publisher is Scholastic Press. There are more Science fiction books.

In [76]:
# Removing the / and - from the Sub-genre
df['Sub-Genre'] = df['Sub-Genre'].str.replace('/', '').str.strip().str.replace('-', '')

In [77]:
vector = TfidfVectorizer(stop_words='english') # Initiating the vectorizer

genre_matrix = vector.fit_transform(df['Sub-Genre']) # Coverting Sub-genre to vector feature

# Training the Nearest Neighbor model
knn = NearestNeighbors(n_neighbors=10, metric='cosine')
knn.fit(genre_matrix)

,n_neighbors,10
,radius,1.0
,algorithm,'auto'
,leaf_size,30
,metric,'cosine'
,p,2
,metric_params,None
,n_jobs,None


In [78]:
def book_recommendation(title = 'The Correspondent: A Novel'):
    
    """
    This funtion will take a title of a book and recommend 10 similiar books

    """
    indx = df[df['Title'] == title].index[0]
    
    distance, indice = knn.kneighbors(genre_matrix[indx])
    
    for i in indice:
        return df['Title'][i].reset_index(drop = True)
    
book_recommendation()

0    Tomorrow, and Tomorrow, and Tomorrow
1                    The Midnight Library
2                               A Admiral
3                               Lost Star
4                             When Window
5                      The Land Blueprint
6                          World and Bone
7                        The Nurse Untold
8              The Correspondent: A Novel
9                             Quiet Clock
Name: Title, dtype: object

In [79]:
book_recommendation('Project Hail Mary')

0     The Dream and Thunder
1               Star of Ice
2            Power Approach
3          The Map of Stars
4       The River Revisited
5    Knight Always Watching
6               Where Clock
7              The War Saga
8            Map Chronicles
9    The Kingdom Reimagined
Name: Title, dtype: object